In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from skimage.feature import corner_harris
from skimage.feature import corner_peaks

## Function Definitions

In [ ]:
# Extract patch-based descriptors for each keypoint
# Each descriptor is a fixed-size (patch_size x patch_size) region around the keypoint
def extract_patch_descriptors(gray, corners, patch_size=11):
    margin = patch_size // 2
    descriptors = []
    valid_corners = []

    for r, c in corners:
        #extract patch around the corner 
        patch = gray[r - margin:r + margin + 1, c - margin:c + margin + 1]

        # ensure the patch is the correct size 
        if patch.shape != (patch_size, patch_size):
            continue

        patch_vector = patch.flatten().astype(np.float32)

        # normalize descriptor: zero mean, unit std
        mean = np.mean(patch_vector)
        std = np.std(patch_vector)

        if std < 1e-6:
            continue  # skip nearly constant patches

        patch_vector = (patch_vector - mean) / std

        #store the descriptor and its corresponding corner location
        descriptors.append(patch_vector)
        valid_corners.append([r, c])

    return np.array(valid_corners), np.array(descriptors)

In [ ]:
def compute_normalized_correlation(desc1, desc2):
    # Compute the normalized correlation between two sets of descriptors
    corr = desc1 @ desc2.T
    return corr

In [ ]:


def compute_euclidean_distances(desc1, desc2):
    # Compute the Euclidean distances between two sets of descriptors 
    dists = np.sqrt(
        np.sum((desc1[:, np.newaxis, :] - desc2[np.newaxis, :, :]) ** 2, axis=2)
    )
    return dists

In [ ]:
# Select best matches based on Euclidean distance
# For each descriptor in image 1, find the closest descriptor in image 2
# Lower distance = more similar descriptors
def best_matches_from_distance(dist_matrix):
    matches = []
    for i in range(dist_matrix.shape[0]):
        j = np.argmin(dist_matrix[i]) # index of closest descriptor in image 2
        score = dist_matrix[i, j]
        matches.append((i, j, score))
    return matches

# Select best matches based on normalized correlation
# For each descriptor in image 1, find the most similar descriptor in image 2
# Higher correlation = more similar descriptors
def best_matches_from_correlation(corr_matrix):
    matches = []
    for i in range(corr_matrix.shape[0]):
        j = np.argmax(corr_matrix[i]) # index of highest similarity
        score = corr_matrix[i, j]
        matches.append((i, j, score))
    return matches

In [ ]:
def select_top_matches_distance(matches, top_k=100):
    matches_sorted = sorted(matches, key=lambda x: x[2])  # smaller distance first
    return matches_sorted[:top_k]

def select_top_matches_correlation(matches, top_k=100):
    matches_sorted = sorted(matches, key=lambda x: x[2], reverse=True)  # larger corr first
    return matches_sorted[:top_k]

In [ ]:
# Convert match indices into corresponding point coordinates
# Each match (i, j) links a keypoint in image 1 to a keypoint in image 2
def matches_to_points(matches, corners1, corners2):
    pts1 = []
    pts2 = []

    for i, j, _ in matches:
        r1, c1 = corners1[i]  # keypoint in image 1 (row, col)
        r2, c2 = corners2[j]  # keypoint in image 2 (row, col)

        # Convert from (row, col) to (x, y) format
        pts1.append([c1, r1])
        pts2.append([c2, r2])

    return np.array(pts1, dtype=np.float32), np.array(pts2, dtype=np.float32)

In [ ]:

# Visualize matches between the two images
def visualize_matches(img1, img2, corners1, corners2, matches, title="Matches"):
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]

    # Create a canvas large enough to hold both images side by side
    canvas_height = max(h1, h2)
    canvas_width = w1 + w2

    canvas = np.zeros((canvas_height, canvas_width, 3), dtype=np.uint8)
    canvas[:h1, :w1] = img1
    canvas[:h2, w1:w1+w2] = img2

    plt.figure(figsize=(14, 7))
    plt.imshow(canvas)

    # Draw lines between matched keypoints
    for i, j, _ in matches:
        r1, c1 = corners1[i]  # keypoint in image 1 (row, col)
        r2, c2 = corners2[j]  # keypoint in image 2 (row, col)

        # Convert to (x, y) and shift second image by width of first image
        x1, y1 = c1, r1
        x2, y2 = c2 + w1, r2

        plt.plot([x1, x2], [y1, y2], linewidth=0.7)
        plt.scatter([x1, x2], [y1, y2], s=10)

    plt.title(title)
    plt.axis("off")
    plt.show()

In [ ]:
# Estimate affine transformation between two sets of matched points
# Affine model: x' = ax + by + c,  y' = dx + ey + f (6 parameters)
# At least 3 point pairs are required to solve for the transformation
def estimate_affine(pts_src, pts_dst):
    
    n = pts_src.shape[0]
    if n < 3:
        raise ValueError("Need at least 3 points to estimate affine transform.")

    A = []
    b = []

    # Build linear system A * params = b
    for (x, y), (xp, yp) in zip(pts_src, pts_dst):
        A.append([x, y, 1, 0, 0, 0])
        A.append([0, 0, 0, x, y, 1])
        b.append(xp)
        b.append(yp)

    A = np.array(A, dtype=np.float32)
    b = np.array(b, dtype=np.float32)

    # Solve using least squares 
    params, _, _, _ = np.linalg.lstsq(A, b, rcond=None)

    # Convert parameters into affine matrix form
    affine = np.array([
        [params[0], params[1], params[2]],
        [params[3], params[4], params[5]]
    ], dtype=np.float32)

    return affine


# Apply affine transformation to a set of points
# Points are converted to homogeneous coordinates (x, y, 1)
def apply_affine(affine, pts):
    
    ones = np.ones((pts.shape[0], 1), dtype=np.float32)
    pts_h = np.hstack([pts, ones])   # (N, 3)

    # Apply transformation: p' = T * p
    transformed = (affine @ pts_h.T).T

    return transformed

In [ ]:
# Compute geometric error (residuals) between transformed points and target points
# Residual = Euclidean distance between estimated position and true match
def compute_residuals(affine, pts_src, pts_dst):
    
    # Apply affine transformation to source points
    pts_transformed = apply_affine(affine, pts_src)

    # Compute squared distance for each point pair
    residuals = np.sum((pts_transformed - pts_dst) ** 2, axis=1)

    return residuals

In [ ]:

# repeatedly sample a small set of matches, estimate an affine model,
# and count how many other matches agree with it (inliers)
def ransac_affine(pts_src, pts_dst, num_iterations=1000, threshold=5.0, sample_size=3):
    best_affine = None
    best_inliers = None
    best_num_inliers = 0
    best_residuals = None

    n = pts_src.shape[0]

    # Need at least sample_size matches to run RANSAC
    if n < sample_size:
        raise ValueError("Not enough matches for RANSAC.")

    # Repeat for a fixed number of random samples
    for _ in range(num_iterations):
        # Randomly select a minimal subset of matches
        idx = np.random.choice(n, sample_size, replace=False)

        sample_src = pts_src[idx]
        sample_dst = pts_dst[idx]

        # Estimate affine transform from the sampled matches
        try:
            affine = estimate_affine(sample_src, sample_dst)
        except:
            continue

        # Measure how well the estimated model fits all matches
        residuals = compute_residuals(affine, pts_src, pts_dst)

        # Matches with residual below the threshold are considered inliers
        inliers = residuals < threshold
        num_inliers = np.sum(inliers)

        # Keep the model with the largest number of inliers
        if num_inliers > best_num_inliers:
            best_num_inliers = num_inliers
            best_affine = affine
            best_inliers = inliers
            best_residuals = residuals

    # Re-estimate the affine transformation using all inliers of the best model
    if best_inliers is not None and np.sum(best_inliers) >= 3:
        best_affine = estimate_affine(pts_src[best_inliers], pts_dst[best_inliers])
        best_residuals = compute_residuals(best_affine, pts_src, pts_dst)

    return best_affine, best_inliers, best_residuals

In [ ]:
# Compute Euclidean accuracy score between matched points
def accuracy_score_euclidean(affine, pts_img2, pts_img1):
    
    # Apply affine transformation to points from image 2
    transformed_pts_img2 = apply_affine(affine, pts_img2)

    # Compute Euclidean distance for each pair of corresponding points
    distances = np.sqrt(np.sum((pts_img1 - transformed_pts_img2) ** 2, axis=1))

    # Compute mean distance as overall accuracy score
    mean_distance = np.mean(distances)

    return distances, mean_distance

## Images Loading

In [ ]:
#img_1 = cv2.imread('leftImage.png')
#img_2 = cv2.imread('rightImage.png')

In [ ]:
import tkinter as tk
from tkinter import filedialog
import os


def select_image(title='Select an image'):
    """Open a file dialog and return the selected image path."""
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    path = filedialog.askopenfilename(
        title=title,
        filetypes=[
            ('Image files', '*.png *.jpg *.jpeg *.bmp *.tiff *.tif'),
            ('All files', '*.*')
        ]
    )
    root.destroy()
    if not path:
        raise FileNotFoundError('No image selected — please re-run and pick a file.')
    return path


print('Select the LEFT image ...')
path_img1 = select_image(title='Select LEFT image')
print(f'  ✔  Left image : {os.path.basename(path_img1)}')

print('Select the RIGHT image ...')
path_img2 = select_image(title='Select RIGHT image')
print(f'  ✔  Right image: {os.path.basename(path_img2)}')

img_1 = cv2.imread(path_img1)
img_2 = cv2.imread(path_img2)

if img_1 is None:
    raise IOError(f'Could not load left image: {path_img1}')
if img_2 is None:
    raise IOError(f'Could not load right image: {path_img2}')

print(f'\nLeft  image shape : {img_1.shape}')
print(f'Right image shape : {img_2.shape}')

In [ ]:

img_1 = cv2.cvtColor(img_1, cv2.COLOR_BGR2RGB)
img_2 = cv2.cvtColor(img_2, cv2.COLOR_BGR2RGB)
img_gray_1 = cv2.cvtColor(img_1, cv2.COLOR_RGB2GRAY)
img_gray_2 = cv2.cvtColor(img_2, cv2.COLOR_RGB2GRAY)

plt.figure(figsize=(10, 8))

plt.subplot(2, 2, 1)
plt.imshow(img_1)
plt.title("Original left image")

plt.subplot(2, 2, 2)
plt.imshow(img_2)
plt.title("Original right image")

plt.subplot(2, 2, 3)
plt.imshow(img_gray_1, cmap='gray')
plt.title("Grayscale left image")

plt.subplot(2, 2, 4)
plt.imshow(img_gray_2, cmap='gray')
plt.title("Grayscale right image")

plt.tight_layout()
plt.show()


## Harris Corner Detection

In [ ]:
# Compute Harris corners for both images 
harris_corners_1 = corner_harris(img_gray_1, k=0.04)
harris_corners_2 = corner_harris(img_gray_2, k=0.04)

#visualize Harris corners 
#bright areas correspond to stronger corners 
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(harris_corners_1, cmap='gray')
axes[0].set_title("Harris Response - Image 1")

axes[1].imshow(harris_corners_2, cmap='gray')
axes[1].set_title("Harris Response - Image 2")

for ax in axes:
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Extract corner coordinates from the Harris response
# corner_peaks applies thresholding and non-maximum suppression
# min_distance avoids detecting multiple nearby points for the same corner
# threshold_rel keeps only strong corner responses
corners_1 = corner_peaks(harris_corners_1, min_distance=5, threshold_rel=0.01)
corners_2 = corner_peaks(harris_corners_2, min_distance=5, threshold_rel=0.01)

# Visualize detected corners on top of the original images
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(img_1)
axes[0].scatter(corners_1[:,1], corners_1[:,0], s=10, c='red')
axes[0].set_title(f"Detected Corners - Image 1: {len(corners_1)}")
axes[0].axis("off")

axes[1].imshow(img_2)
axes[1].scatter(corners_2[:,1], corners_2[:,0], s=10, c='red')
axes[1].set_title(f"Detected Corners - Image 2: {len(corners_2)}")
axes[1].axis("off")

plt.tight_layout()
plt.show()


## Patch Descriptor Extraction

Patch sizes investigated: 5, 9, 13, 17, 21

In [ ]:
# Sensitivity analysis: effect of patch size on matching quality
# Metrics: descriptor count, inliers, outliers, inlier ratio,
#          mean Euclidean error, mean inlier distance
patch_sizes = [5, 9, 13, 17, 21]

ps_desc_counts_1 = []
ps_desc_counts_2 = []
ps_inliers        = []
ps_outliers       = []
ps_inlier_ratio   = []
ps_mean_euc_err   = []
ps_mean_inlier_d  = []

print(f"{'Patch':>5} | {'Desc1':>6} | {'Desc2':>6} | {'Inliers':>7} | {'Outliers':>8} | {'InlRatio':>8} | {'MeanEucErr':>10} | {'MeanInlDist':>11}")
print('-' * 80)

for ps in patch_sizes:
    _raw_c1 = corner_peaks(corner_harris(img_gray_1, k=0.04), min_distance=5, threshold_rel=0.01)
    _raw_c2 = corner_peaks(corner_harris(img_gray_2, k=0.04), min_distance=5, threshold_rel=0.01)
    _c1, _d1 = extract_patch_descriptors(img_gray_1, _raw_c1, patch_size=ps)
    _c2, _d2 = extract_patch_descriptors(img_gray_2, _raw_c2, patch_size=ps)
    _dist  = compute_euclidean_distances(_d1, _d2)
    _mlist = best_matches_from_distance(_dist)
    _top   = select_top_matches_distance(_mlist, top_k=100)
    _p1, _p2 = matches_to_points(_top, _c1, _c2)
    _affine, _inliers, _ = ransac_affine(_p2, _p1, num_iterations=1000, threshold=25.0, sample_size=3)
    _n_inl  = int(np.sum(_inliers))
    _n_outl = len(_inliers) - _n_inl
    _ratio  = _n_inl / len(_inliers)
    _, _mean_euc = accuracy_score_euclidean(_affine, _p2, _p1)
    _, _mean_inl = accuracy_score_euclidean(_affine, _p2[_inliers], _p1[_inliers])
    ps_desc_counts_1.append(len(_d1))
    ps_desc_counts_2.append(len(_d2))
    ps_inliers.append(_n_inl)
    ps_outliers.append(_n_outl)
    ps_inlier_ratio.append(_ratio)
    ps_mean_euc_err.append(_mean_euc)
    ps_mean_inlier_d.append(_mean_inl)
    print(f"  {ps:>3}  | {len(_d1):>6} | {len(_d2):>6} | {_n_inl:>7} | {_n_outl:>8} | {_ratio:>8.3f} | {_mean_euc:>10.2f} | {_mean_inl:>11.2f}")

# Plot
fig_ps, axes = plt.subplots(2, 3, figsize=(16, 9))
fig_ps.suptitle('Sensitivity Analysis: Patch Size', fontsize=14, fontweight='bold')
x = np.arange(len(patch_sizes)); w = 0.35

ax = axes[0, 0]
b1 = ax.bar(x - w/2, ps_desc_counts_1, w, label='Image 1')
b2 = ax.bar(x + w/2, ps_desc_counts_2, w, label='Image 2')
ax.set_title('Descriptor Count'); ax.set_xlabel('Patch Size'); ax.set_ylabel('# Descriptors')
ax.set_xticks(x); ax.set_xticklabels(patch_sizes); ax.legend()
for b in list(b1)+list(b2):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+2, str(int(b.get_height())), ha='center', va='bottom', fontsize=7)

ax = axes[0, 1]
ax.bar(x - w/2, ps_inliers,  w, label='Inliers',  color='steelblue')
ax.bar(x + w/2, ps_outliers, w, label='Outliers', color='tomato')
ax.set_title('Inliers vs Outliers (top-K=100)'); ax.set_xlabel('Patch Size'); ax.set_ylabel('# Matches')
ax.set_xticks(x); ax.set_xticklabels(patch_sizes); ax.legend()

ax = axes[0, 2]
ax.plot(patch_sizes, ps_inlier_ratio, marker='o', color='steelblue')
ax.set_title('Inlier Ratio'); ax.set_xlabel('Patch Size'); ax.set_ylabel('Inlier Ratio')
ax.set_xticks(patch_sizes); ax.grid(True, linestyle='--', alpha=0.5)
for px, py in zip(patch_sizes, ps_inlier_ratio):
    ax.annotate(f'{py:.3f}', (px, py), textcoords='offset points', xytext=(0,6), ha='center', fontsize=8)

ax = axes[1, 0]
ax.plot(patch_sizes, ps_mean_euc_err, marker='s', color='tomato')
ax.set_title('Mean Euclidean Error (all matches)'); ax.set_xlabel('Patch Size'); ax.set_ylabel('Mean Error (px)')
ax.set_xticks(patch_sizes); ax.grid(True, linestyle='--', alpha=0.5)
for px, py in zip(patch_sizes, ps_mean_euc_err):
    ax.annotate(f'{py:.1f}', (px, py), textcoords='offset points', xytext=(0,6), ha='center', fontsize=8)

ax = axes[1, 1]
ax.plot(patch_sizes, ps_mean_inlier_d, marker='^', color='seagreen')
ax.set_title('Mean Inlier Distance'); ax.set_xlabel('Patch Size'); ax.set_ylabel('Mean Inlier Dist (px)')
ax.set_xticks(patch_sizes); ax.grid(True, linestyle='--', alpha=0.5)
for px, py in zip(patch_sizes, ps_mean_inlier_d):
    ax.annotate(f'{py:.2f}', (px, py), textcoords='offset points', xytext=(0,6), ha='center', fontsize=8)

ax = axes[1, 2]; ax2 = ax.twinx()
l1, = ax.plot(patch_sizes, ps_inlier_ratio, marker='o', color='steelblue', label='Inlier Ratio')
l2, = ax2.plot(patch_sizes, ps_mean_euc_err, marker='s', linestyle='--', color='tomato', label='Mean Euc Error')
ax.set_title('Summary: Ratio vs Error'); ax.set_xlabel('Patch Size')
ax.set_ylabel('Inlier Ratio', color='steelblue'); ax2.set_ylabel('Mean Euc Error (px)', color='tomato')
ax.set_xticks(patch_sizes); ax.legend(handles=[l1, l2], loc='center right')

plt.tight_layout()
plt.show()

# Use patch_size=21 for the rest of the pipeline
patch_size = 21
print(f'\nSelected patch_size = {patch_size} for the rest of the pipeline')

In [ ]:

corners_1, descriptors_1 = extract_patch_descriptors(img_gray_1, corners_1, patch_size=patch_size)
corners_2, descriptors_2 = extract_patch_descriptors(img_gray_2, corners_2, patch_size=patch_size)

print("Image 1 descriptors:", descriptors_1.shape)
print("Image 2 descriptors:", descriptors_2.shape)

## Descriptor Distance Computation

In [ ]:
corr_matrix = compute_normalized_correlation(descriptors_1, descriptors_2)
print(corr_matrix.shape)

dist_matrix = compute_euclidean_distances(descriptors_1, descriptors_2)
print(dist_matrix.shape)

In [ ]:
matches_euc = best_matches_from_distance(dist_matrix)
matches_corr = best_matches_from_correlation(corr_matrix)

print("Euclidean candidate matches:", len(matches_euc))
print("Correlation candidate matches:", len(matches_corr))
print("First 5 Euclidean matches:", matches_euc[:5])
print("First 5 Correlation matches:", matches_corr[:5])

## Match Selection

top_k investigated: 50, 100, 150, 200

In [ ]:
# Sensitivity analysis: effect of top_k on match quality after RANSAC
top_k_values    = [50, 100, 150, 200]

tk_euc_counts    = []
tk_inliers       = []
tk_outliers      = []
tk_inlier_ratio  = []
tk_mean_euc_err  = []
tk_mean_inlier_d = []

print(f"{'top_k':>5} | {'Matches':>7} | {'Inliers':>7} | {'Outliers':>8} | {'InlRatio':>8} | {'MeanEucErr':>10} | {'MeanInlDist':>11}")
print('-' * 75)

for k in top_k_values:
    _top = select_top_matches_distance(matches_euc, top_k=k)
    _p1, _p2 = matches_to_points(_top, corners_1, corners_2)
    _affine, _inliers, _ = ransac_affine(_p2, _p1, num_iterations=1000, threshold=25.0, sample_size=3)
    _n_inl  = int(np.sum(_inliers))
    _n_outl = len(_inliers) - _n_inl
    _ratio  = _n_inl / len(_inliers)
    _, _mean_euc = accuracy_score_euclidean(_affine, _p2, _p1)
    _, _mean_inl = accuracy_score_euclidean(_affine, _p2[_inliers], _p1[_inliers])
    tk_euc_counts.append(len(_top))
    tk_inliers.append(_n_inl)
    tk_outliers.append(_n_outl)
    tk_inlier_ratio.append(_ratio)
    tk_mean_euc_err.append(_mean_euc)
    tk_mean_inlier_d.append(_mean_inl)
    print(f"  {k:>3}  | {len(_top):>7} | {_n_inl:>7} | {_n_outl:>8} | {_ratio:>8.3f} | {_mean_euc:>10.2f} | {_mean_inl:>11.2f}")

# Plot
fig_tk, axes = plt.subplots(2, 3, figsize=(16, 9))
fig_tk.suptitle('Sensitivity Analysis: Top-K Matches', fontsize=14, fontweight='bold')
x = np.arange(len(top_k_values)); w = 25

ax = axes[0, 0]
ax.bar(top_k_values, tk_euc_counts, width=w, color='steelblue')
ax.set_title('Total Matches Selected'); ax.set_xlabel('top_k'); ax.set_ylabel('# Matches')
ax.set_xticks(top_k_values)
for px, py in zip(top_k_values, tk_euc_counts):
    ax.text(px, py+0.5, str(py), ha='center', va='bottom', fontsize=8)

ax = axes[0, 1]
xb = np.arange(len(top_k_values)); wb = 0.35
ax.bar(xb - wb/2, tk_inliers,  wb, label='Inliers',  color='steelblue')
ax.bar(xb + wb/2, tk_outliers, wb, label='Outliers', color='tomato')
ax.set_title('Inliers vs Outliers'); ax.set_xlabel('top_k'); ax.set_ylabel('# Matches')
ax.set_xticks(xb); ax.set_xticklabels(top_k_values); ax.legend()

ax = axes[0, 2]
ax.plot(top_k_values, tk_inlier_ratio, marker='o', color='steelblue')
ax.set_title('Inlier Ratio'); ax.set_xlabel('top_k'); ax.set_ylabel('Inlier Ratio')
ax.set_xticks(top_k_values); ax.grid(True, linestyle='--', alpha=0.5)
for px, py in zip(top_k_values, tk_inlier_ratio):
    ax.annotate(f'{py:.3f}', (px, py), textcoords='offset points', xytext=(0,6), ha='center', fontsize=8)

ax = axes[1, 0]
ax.plot(top_k_values, tk_mean_euc_err, marker='s', color='tomato')
ax.set_title('Mean Euclidean Error (all matches)'); ax.set_xlabel('top_k'); ax.set_ylabel('Mean Error (px)')
ax.set_xticks(top_k_values); ax.grid(True, linestyle='--', alpha=0.5)
for px, py in zip(top_k_values, tk_mean_euc_err):
    ax.annotate(f'{py:.1f}', (px, py), textcoords='offset points', xytext=(0,6), ha='center', fontsize=8)

ax = axes[1, 1]
ax.plot(top_k_values, tk_mean_inlier_d, marker='^', color='seagreen')
ax.set_title('Mean Inlier Distance'); ax.set_xlabel('top_k'); ax.set_ylabel('Mean Inlier Dist (px)')
ax.set_xticks(top_k_values); ax.grid(True, linestyle='--', alpha=0.5)
for px, py in zip(top_k_values, tk_mean_inlier_d):
    ax.annotate(f'{py:.2f}', (px, py), textcoords='offset points', xytext=(0,6), ha='center', fontsize=8)

ax = axes[1, 2]; ax2 = ax.twinx()
l1, = ax.plot(top_k_values, tk_inlier_ratio, marker='o', color='steelblue', label='Inlier Ratio')
l2, = ax2.plot(top_k_values, tk_mean_euc_err, marker='s', linestyle='--', color='tomato', label='Mean Euc Error')
ax.set_title('Summary: Ratio vs Error'); ax.set_xlabel('top_k')
ax.set_ylabel('Inlier Ratio', color='steelblue'); ax2.set_ylabel('Mean Euc Error (px)', color='tomato')
ax.set_xticks(top_k_values); ax.legend(handles=[l1, l2], loc='center right')

plt.tight_layout()
plt.show()

# Use top_k=100 for the rest of the pipeline
top_k = 100
top_matches_euc  = select_top_matches_distance(matches_euc, top_k=top_k)
top_matches_corr = select_top_matches_correlation(matches_corr, top_k=top_k)
print(f'\nSelected top_k = {top_k}')
print('Top Euclidean matches:', len(top_matches_euc))
print('Top Correlation matches:', len(top_matches_corr))

In [ ]:
pts1_euc, pts2_euc   = matches_to_points(top_matches_euc,  corners_1, corners_2)
pts1_corr, pts2_corr = matches_to_points(top_matches_corr, corners_1, corners_2)
print(pts1_euc.shape, pts2_euc.shape)


In [ ]:
visualize_matches(img_1, img_2, corners_1, corners_2, top_matches_euc,  title='Top Euclidean Matches')
visualize_matches(img_1, img_2, corners_1, corners_2, top_matches_corr, title='Top Correlation Matches')


## RANSAC

In [ ]:
# Run RANSAC on Euclidean matches
best_affine_euc, inliers_euc, residuals_euc = ransac_affine(
    pts2_euc, pts1_euc,
    num_iterations=1000,
    threshold=25.0,
    sample_size=3
)
# Keep backward-compatible aliases used by the rest of the pipeline
best_affine = best_affine_euc
inliers     = inliers_euc
residuals   = residuals_euc

num_inliers  = np.sum(inliers_euc)
num_outliers = len(inliers_euc) - num_inliers
avg_inlier_residual = np.mean(residuals_euc[inliers_euc]) if num_inliers > 0 else None

print('=== RANSAC (Euclidean matches) ===')
print('Inliers:', num_inliers)
print('Outliers:', num_outliers)
print('Average inlier residual:', avg_inlier_residual)
print('Inlier ratio:', num_inliers / len(inliers_euc))

inlier_matches_euc = [m for m, keep in zip(top_matches_euc, inliers_euc) if keep]
visualize_matches(img_1, img_2, corners_1, corners_2, inlier_matches_euc, title='RANSAC Inlier Matches (Euclidean)')

# Run RANSAC on Correlation matches
best_affine_corr, inliers_corr, residuals_corr = ransac_affine(
    pts2_corr, pts1_corr,
    num_iterations=1000,
    threshold=25.0,
    sample_size=3
)

num_inliers_corr  = np.sum(inliers_corr)
num_outliers_corr = len(inliers_corr) - num_inliers_corr
avg_res_corr = np.mean(residuals_corr[inliers_corr]) if num_inliers_corr > 0 else None

print('\n=== RANSAC (Correlation matches) ===')
print('Inliers:', num_inliers_corr)
print('Outliers:', num_outliers_corr)
print('Average inlier residual:', avg_res_corr)
print('Inlier ratio:', num_inliers_corr / len(inliers_corr))

inlier_matches_corr = [m for m, keep in zip(top_matches_corr, inliers_corr) if keep]
visualize_matches(img_1, img_2, corners_1, corners_2, inlier_matches_corr, title='RANSAC Inlier Matches (Correlation)')

In [ ]:
def warp_and_stitch_affine(img1, img2, affine):
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]

    # corners of image 2
    corners_img2 = np.array([
        [0, 0],
        [w2, 0],
        [w2, h2],
        [0, h2]
    ], dtype=np.float32)

    # transform corners of image 2
    transformed_corners_img2 = apply_affine(affine, corners_img2)

    # corners of image 1 (already in panorama reference frame)
    corners_img1 = np.array([
        [0, 0],
        [w1, 0],
        [w1, h1],
        [0, h1]
    ], dtype=np.float32)

    # combine all corners
    all_corners = np.vstack((corners_img1, transformed_corners_img2))

    x_min, y_min = np.floor(all_corners.min(axis=0)).astype(int)
    x_max, y_max = np.ceil(all_corners.max(axis=0)).astype(int)

    # translation to keep coordinates positive
    tx = -x_min if x_min < 0 else 0
    ty = -y_min if y_min < 0 else 0

    # output canvas size
    out_w = x_max - x_min
    out_h = y_max - y_min

    # add translation to affine matrix
    affine_translated = affine.copy()
    affine_translated[0, 2] += tx
    affine_translated[1, 2] += ty

    # warp image 2
    warped_img2 = cv2.warpAffine(img2, affine_translated, (out_w, out_h))

    # create panorama canvas
    panorama = warped_img2.copy()

    # paste image 1 into panorama
    panorama[ty:ty + h1, tx:tx + w1] = img1

    return panorama, warped_img2, affine_translated


panorama, warped_img2, best_affine_translated = warp_and_stitch_affine(img_1, img_2, best_affine)

plt.figure(figsize=(20, 10))
plt.imshow(panorama)
plt.title("Stitched Panorama")
plt.axis("off")
plt.show()

## Accuracy Evaluation

In [ ]:
# Compute accuracy over all matches
distances, mean_distance = accuracy_score_euclidean(best_affine, pts2_euc, pts1_euc)
print('Mean Euclidean accuracy score:', mean_distance)

# Compute accuracy only on inlier matches
inlier_pts2 = pts2_euc[inliers]
inlier_pts1 = pts1_euc[inliers]
inlier_distances, mean_inlier_distance = accuracy_score_euclidean(best_affine, inlier_pts2, inlier_pts1)
print('Mean inlier Euclidean distance:', mean_inlier_distance)
print('Number of inliers:', np.sum(inliers))
print('Number of outliers:', len(inliers) - np.sum(inliers))
print('Average inlier residual:', np.mean(inlier_distances))


Save Results


In [ ]:

import datetime

# Create results folder
results_dir = 'results'
os.makedirs(results_dir, exist_ok=True)

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

# 1. Stitched panorama
panorama_bgr = cv2.cvtColor(panorama, cv2.COLOR_RGB2BGR)
cv2.imwrite(os.path.join(results_dir, f'panorama_{timestamp}.png'), panorama_bgr)
print(f'Panorama saved        → results/panorama_{timestamp}.png')

# 2. Harris corners
fig_harris, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(img_1); axes[0].scatter(corners_1[:,1], corners_1[:,0], s=10, c='red')
axes[0].set_title(f'Detected Corners - Image 1: {len(corners_1)}'); axes[0].axis('off')
axes[1].imshow(img_2); axes[1].scatter(corners_2[:,1], corners_2[:,0], s=10, c='red')
axes[1].set_title(f'Detected Corners - Image 2: {len(corners_2)}'); axes[1].axis('off')
fig_harris.tight_layout()
fig_harris.savefig(os.path.join(results_dir, f'harris_corners_{timestamp}.png'), dpi=150, bbox_inches='tight')
plt.close(fig_harris)
print(f'Harris corners saved  → results/harris_corners_{timestamp}.png')

# 3. Inlier matches
fig_matches, ax = plt.subplots(figsize=(14, 7))
h1, w1 = img_1.shape[:2]; h2, w2 = img_2.shape[:2]
canvas = np.zeros((max(h1, h2), w1 + w2, 3), dtype=np.uint8)
canvas[:h1, :w1] = img_1; canvas[:h2, w1:] = img_2
ax.imshow(canvas)
for i, j, _ in inlier_matches_euc:
    r1, c1 = corners_1[i]; r2, c2 = corners_2[j]
    ax.plot([c1, c2 + w1], [r1, r2], linewidth=0.7)
    ax.scatter([c1, c2 + w1], [r1, r2], s=10)
ax.set_title('RANSAC Inlier Matches (Euclidean)'); ax.axis('off')
fig_matches.tight_layout()
fig_matches.savefig(os.path.join(results_dir, f'inlier_matches_{timestamp}.png'), dpi=150, bbox_inches='tight')
plt.close(fig_matches)
print(f'Inlier matches saved  → results/inlier_matches_{timestamp}.png')

# 4. Sensitivity analysis plots
fig_ps.savefig(os.path.join(results_dir, f'sensitivity_patch_size_{timestamp}.png'), dpi=150, bbox_inches='tight')
print(f'Patch size plot saved → results/sensitivity_patch_size_{timestamp}.png')

fig_tk.savefig(os.path.join(results_dir, f'sensitivity_top_k_{timestamp}.png'), dpi=150, bbox_inches='tight')
print(f'Top-K plot saved      → results/sensitivity_top_k_{timestamp}.png')

print(f'\nAll results saved in → ./{results_dir}/')